# Отчёт по лабораторной работе №1

## Введение в Spring Boot

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

## Цель работы

Получить практические навыки работы со Spring Boot, создать веб-приложение
для хранения заметок и изучить обработку HTTP-запросов.

## Задание

1. Изучить теоретический материал.
2. Разработать приложение для добавления, просмотра, изменения и удаления сообщений.
3. Реализовать два дополнительных метода: очистку всех сообщений и подсчёт их количества.
4. Проверить работу приложения и подготовить отчёт.

## Ход работы

### 1. Создание проекта

Создан Gradle-проект `messages`. Использованы Java 17, Gradle 8.12 и Spring Boot 3.2.3.
Для веб-приложения подключён `spring-boot-starter-web`, для тестирования
используется `spring-boot-starter-test`.

```text
lab1/
├── build.gradle
├── settings.gradle
├── gradlew
├── gradlew.bat
├── gradle/wrapper/
└── src/
    ├── main/
    │   ├── java/ru/bsuedu/cad/lab/
    │   │   ├── MessagesApplication.java
    │   │   └── MessageController.java
    │   └── resources/application.properties
    └── test/java/ru/bsuedu/cad/lab/
        └── MessageControllerTest.java
```

### 2. Настройка Gradle

В `build.gradle` указаны плагины, версия Java и зависимости.
Плагин Spring Boot позволяет запускать приложение командой `bootRun`
и собирать исполняемый JAR командой `bootJar`.

```groovy
plugins {
    id 'java'
    id 'org.springframework.boot' version '3.2.3'
    id 'io.spring.dependency-management' version '1.1.4'
}

group = 'ru.bsuedu.cad.lab'
version = '1.0'

java {
    sourceCompatibility = JavaVersion.VERSION_17
    targetCompatibility = JavaVersion.VERSION_17
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

tasks.withType(JavaCompile).configureEach {
    options.encoding = 'UTF-8'
    options.compilerArgs.add('-parameters')
}
```

### 3. Точка входа приложения

Класс `MessagesApplication` запускает приложение.
Аннотация `@SpringBootApplication` включает автоматическую настройку
Spring Boot и поиск компонентов.

```java
package ru.bsuedu.cad.lab;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class MessagesApplication {

    public static void main(String[] args) {
        SpringApplication.run(MessagesApplication.class, args);
    }
}
```

### 4. Контроллер сообщений

В `MessageController` создан список `userMessages` для хранения заметок.
Аннотация `@RestController` обозначает обработчик HTTP-запросов.
`@RequestBody` получает текст сообщения из тела запроса,
а `@PathVariable` получает индекс из адреса.

Индексы начинаются с нуля. Сообщения хранятся в памяти и удаляются при остановке
приложения. После удаления заметки индексы оставшихся элементов сдвигаются.
Методы синхронизированы, а при получении списка возвращается его копия.

```java
package ru.bsuedu.cad.lab;

import org.springframework.web.bind.annotation.DeleteMapping;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PathVariable;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PutMapping;
import org.springframework.web.bind.annotation.RequestBody;
import org.springframework.web.bind.annotation.RestController;

import java.util.ArrayList;
import java.util.List;

@RestController
public class MessageController {

    private final List<String> userMessages = new ArrayList<>();

    @GetMapping("/")
    public String helloWorld() {
        return "Привет, мир!";
    }

    @GetMapping("/messages")
    public synchronized List<String> getAllMessages() {
        return new ArrayList<>(userMessages);
    }

    @PostMapping("/messages")
    public synchronized String publishMessage(@RequestBody String message) {
        userMessages.add(message);
        return "Сообщение успешно опубликовано!";
    }

    @PutMapping("/messages/{index}")
    public synchronized String updateMessage(@PathVariable int index,
                                             @RequestBody String message) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.set(index, message);
            return "Сообщение успешно изменено!";
        }
        return "Сообщение с индексом " + index + " не найдено";
    }

    @DeleteMapping("/messages/{index}")
    public synchronized String deleteMessage(@PathVariable int index) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.remove(index);
            return "Сообщение успешно удалено!";
        }
        return "Сообщение с индексом " + index + " не найдено";
    }

    @DeleteMapping("/messages")
    public synchronized String clearAllMessages() {
        userMessages.clear();
        return "Все сообщения успешно удалены!";
    }

    @GetMapping("/messages/count")
    public synchronized int countMessages() {
        return userMessages.size();
    }
}
```

### 5. Запросы приложения

| Метод | Адрес | Назначение |
| --- | --- | --- |
| GET | `/` | Приветствие |
| GET | `/messages` | Получить список сообщений |
| POST | `/messages` | Добавить сообщение |
| PUT | `/messages/{index}` | Изменить сообщение по индексу |
| DELETE | `/messages/{index}` | Удалить сообщение по индексу |
| DELETE | `/messages` | Очистить все сообщения |
| GET | `/messages/count` | Получить количество сообщений |

При добавлении и изменении передаётся текст с типом `text/plain; charset=utf-8`.

### 6. Запуск приложения

Приложение запущено локально из папки Gradle-проекта командой:

```powershell
.\gradlew.bat bootRun
```

При обращении к корневому адресу сервер возвращает `Привет, мир!`.
Для снимков использован локальный порт 18181. Обычный запуск через `bootRun`
использует порт 8080.

<img src="https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/greeting.jpg" alt="Рисунок 1. Ответ приложения на GET /" width="1000">

*Рисунок 1. Ответ приложения на GET /*

### 7. Проверка основных операций

Сначала получен пустой список. Затем добавлена заметка `Первая заметка`,
её текст изменён на `Изменённая заметка`, после чего заметка удалена.
После каждой операции проверялось содержимое списка.

Результаты запросов к локальному приложению:

```text
GET / -> HTTP 200: Привет, мир!
GET /messages -> HTTP 200: []
POST /messages -> HTTP 200: Сообщение успешно опубликовано!
GET /messages -> HTTP 200: ["Первая заметка"]
PUT /messages/0 -> HTTP 200: Сообщение успешно изменено!
GET /messages -> HTTP 200: ["Изменённая заметка"]
DELETE /messages/0 -> HTTP 200: Сообщение успешно удалено!
GET /messages -> HTTP 200: []
```

Дополнительно добавлены две заметки и открыт адрес `/messages` в браузере.
Сервер вернул список в формате JSON.

<img src="https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/messages.jpg" alt="Рисунок 2. Список из двух сообщений" width="1000">

*Рисунок 2. Список из двух сообщений*

### 8. Дополнительные задания

**Пункт 1. Очистка списка**

Метод `clearAllMessages()` удаляет все сообщения. Для него используется запрос
`DELETE /messages`.

```java
@DeleteMapping("/messages")
public synchronized String clearAllMessages() {
    userMessages.clear();
    return "Все сообщения успешно удалены!";
}
```

**Пункт 6. Подсчёт сообщений**

Метод `countMessages()` возвращает размер списка. Для него используется запрос
`GET /messages/count`.

```java
@GetMapping("/messages/count")
public synchronized int countMessages() {
    return userMessages.size();
}
```

Для проверки добавлены две заметки. До очистки количество сообщений равно 2,
после очистки равно 0. Повторная очистка пустого списка также допустима.

```text
POST /messages -> HTTP 200: Сообщение успешно опубликовано!
POST /messages -> HTTP 200: Сообщение успешно опубликовано!
GET /messages -> HTTP 200: ["Заметка один","Заметка два"]
GET /messages/count -> HTTP 200: 2
DELETE /messages -> HTTP 200: Все сообщения успешно удалены!
GET /messages -> HTTP 200: []
GET /messages/count -> HTTP 200: 0
DELETE /messages -> HTTP 200: Все сообщения успешно удалены!
```

### 9. Проверка неверного индекса

При попытке изменить или удалить отсутствующее сообщение возвращается текст
`Сообщение с индексом N не найдено`, список при этом не изменяется.
Код ответа 200 сохранён по примеру из методички.

```text
PUT /messages/-1 -> HTTP 200: Сообщение с индексом -1 не найдено
DELETE /messages/10 -> HTTP 200: Сообщение с индексом 10 не найдено
GET /messages -> HTTP 200: []
```

### 10. Автоматическая проверка

Проверка выполнена локально командой:

```powershell
.\gradlew.bat test bootJar
```

Проверены приветствие, получение списка, добавление, изменение, удаление,
очистка, подсчёт сообщений и обращения по неверным индексам.
Все 10 тестовых случаев завершились успешно. Исполняемый JAR собран.

<img src="https://raw.githubusercontent.com/kir1lka/cad-2025/main/sem2/lab1/images/tests.jpg" alt="Рисунок 3. Результаты тестов Gradle" width="1000">

*Рисунок 3. Результаты тестов Gradle*

## Вывод

В ходе лабораторной работы создано веб-приложение Spring Boot для хранения заметок.
Реализованы добавление, просмотр, изменение и удаление сообщений.
Выполнены дополнительные задания по очистке списка и подсчёту сообщений.
Работа приложения проверена HTTP-запросами и автоматическими тестами.

## Материалы

1. Методичка «Работа 1 Spring.pdf».
2. [Документация Spring Boot 3.2.3](https://docs.spring.io/spring-boot/docs/3.2.3/reference/html/getting-started.html).